<a href="https://colab.research.google.com/github/Prabhuarasu/data-scince/blob/main/Copy_of_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data uploading/downloading

In [ ]:
# option #1
from google.colab import files
uploaded = files.upload()

In [ ]:
# option #2
!wget https://raw.githubusercontent.com/ziafaq/genai/refs/heads/main/HousesInfo.txt

# Data reading

In [ ]:
data_file = '/content/HousesInfo.txt'

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
#session_path = /content/
#filename = HousesInfo.txt

In [ ]:
df = pd.read_csv(data_file, sep=' ', header=None)

In [ ]:
df

In [ ]:
df.columns = ['bedrooms', 'bathrooms', 'area', 'zipcode', 'price']
df.head(2)

In [ ]:
# option 3
df2 = pd.read_csv('https://raw.githubusercontent.com/ziafaq/genai/refs/heads/main/HousesInfo.txt',
                 sep=' ', header=None, names= ['bedrooms', 'bathrooms', 'area', 'zipcode', 'price'])
df2.head(2)

In [ ]:
# option 4
def load_data_attributes(inputPath)-> pd.DataFrame():
    cols = ['bedrooms', 'bathrooms', 'area', 'zipcode', 'price']
    df = pd.read_csv(inputPath,
                 sep=' ', header=None, names = cols)
    return df

In [ ]:
data = load_data_attributes(data_file)
data.head(2)

In [ ]:
# get the columnlist
data.columns

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data[data.duplicated()]

In [ ]:
data[data.duplicated(keep=False)]

In [ ]:
data.shape, data.index

In [ ]:
data.drop_duplicates(inplace=True, ignore_index=True)

In [ ]:
data.shape, data.index

In [ ]:
data.nunique()

In [ ]:
data.info()

In [ ]:
data['zipcode'].value_counts()

In [ ]:
data = load_data_attributes(data_file)
data.drop_duplicates(inplace=True, ignore_index=True)
data.info()

In [ ]:
# to reduce
def filter_less_unique_values(df, col, min_val, handle='drop')->pd.DataFrame:
  dff = df.copy()
  zipcodes = dff[col].value_counts().keys().tolist()
  counts = dff[col].value_counts().tolist()

  for (zipcodecol,count) in zip(zipcodes,counts):
    #min_val
    #handle = drop, replace
    if count < min_val:
      idxs = dff[dff[col] == zipcodecol].index
      if handle == 'drop':
        dff.drop(idxs, inplace=True)
      if handle == 'replace':
        for idx in idxs:
          dff.loc[idx,col] = 'OTH'
  return dff


In [ ]:
data = load_data_attributes(data_file)
data.drop_duplicates(inplace=True, ignore_index=True)

# fn call
data2 = filter_less_unique_values(data,'zipcode',10)
data2.info(), data2['zipcode'].nunique(), data2['zipcode'].value_counts()

In [ ]:
data = load_data_attributes(data_file)
data.drop_duplicates(inplace=True, ignore_index=True)

# fn call 'replace'
data3 = filter_less_unique_values(data,'zipcode',10,'replace')
data3.info(), data3['zipcode'].nunique(), data3['zipcode'].value_counts()

In [ ]:
data.info(),data['zipcode'].nunique(), data['zipcode'].value_counts()

In [ ]:
#90803 = 'CA'
#95006

# another option
!pip install uszipcode

In [ ]:
#!pip install --force-reinstall uszipcode==0.2.6 sqlalchemy-mate==1.4.0
!pip install sqlalchemy-mate==1.4.28.4

In [ ]:
data = load_data_attributes(data_file)
data.drop_duplicates(inplace=True, ignore_index=True)

In [ ]:
from uszipcode import SearchEngine

search = SearchEngine()


def get_city(zipcode):
  res = search.by_zipcode(str(zipcode))
  if res:
    if res.state:
      return res.state
  return 'OTH'

# feature engg
data['state'] = data['zipcode'].apply(get_city)
data.head()


In [ ]:
data.state.value_counts()

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
data = load_data_attributes(data_file)
data.drop_duplicates(inplace=True, ignore_index=True)
data_copy = data.copy()

In [ ]:
data.nunique()

In [ ]:
ign_col = []
cat_col = ['zipcode']
num_col = ['bedrooms', 'bathrooms', 'area']


In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc_cols = ohe.fit_transform(data[cat_col])

enc_df = pd.DataFrame(enc_cols,columns=ohe.get_feature_names_out())

data = pd.concat([data.drop(cat_col,axis=1),enc_df],axis=1)

data



In [ ]:
scaler = MinMaxScaler()
scaled_data = pd.DataFrame(scaler.fit_transform(data[num_col]), columns=num_col)

data = pd.concat([scaled_data, data.drop(num_col,axis=1)],axis=1)

data

In [ ]:
#split data
tgt_col = ['price']

X = data.drop(columns=tgt_col)
y = data[tgt_col]

train_val_X, test_X, train_val_y, test_y = train_test_split(X,y,test_size=0.1, random_state=42)

train_X, val_X, train_y, val_y = train_test_split(train_val_X,train_val_y,test_size=0.1, random_state=42)

print(train_val_X.shape, test_X.shape, val_X.shape, data.shape)


In [ ]:
model = LinearRegression()

In [ ]:
model.fit(train_X,train_y)

In [ ]:
pred_val_y = model.predict(val_X)
print('val Mape:',mean_absolute_percentage_error(val_y, pred_val_y))

pred_test_y = model.predict(test_X)
print('test Mape:',mean_absolute_percentage_error(test_y, pred_test_y))


In [ ]:
import keras

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.optimizers import Adam
from keras.losses import MeanSquaredError

In [ ]:
def create_nn(dim):
  model = Sequential()
  model.add(Dense(128, input_dim=dim, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(32, activation='relu'))
  model.add(Dense(1, activation='linear'))
  return model




In [ ]:
train_X.shape[1]

In [ ]:
1e-2, 1/100

In [ ]:
model = create_nn(train_X.shape[1])
opt = Adam(learning_rate=1e-2)
model.compile(loss='mean_absolute_percentage_error', optimizer = opt)

from keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10)

model.fit(x=train_X, y=train_y,validation_data=(val_X,val_y),
          epochs=100, batch_size=10, callbacks=[early_stop])

In [ ]:
pred_val_y = model.predict(val_X)
print('val Mape:',mean_absolute_percentage_error(val_y, pred_val_y))

pred_test_y = model.predict(test_X)
print('test Mape:',mean_absolute_percentage_error(test_y, pred_test_y))


In [ ]:
model.save('hp_nn_model.keras')

In [ ]:
from keras.models import load_model
import joblib

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(ohe, 'encoder.pkl')


In [ ]:
!pip install streamlit

In [ ]:
import streamlit

In [ ]:
%%writefile app.py

'''
import streamlit as st
import  numpy as np
import pandas as pd

import joblib
from keras.models import load_model

import warnings
warnings.filterwarnings('ignore')

st.title('Housing PricePrediction')

def predict():
  pass

fp = '/content/HousesInfo.txt'
df = pd.read_csv(fp, sep=' ', header=None)
df.columns = ['bedrooms', 'bathrooms', 'area', 'zipcode', 'price']

selected_values = {}
for column in df.drop(columns=['price']).columns:
  selected_values[column] = st.number_input(
      f'Select a value for {column}')
'''
import streamlit as st
import numpy as np
import pandas as pd

import joblib
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.models import load_model

# Title of the app
st.title("Housing Price Prediction")

# Instructions
st.write("""Select the housing features you'd like to predict:""")

# Load the pickled models and keras model
pscaler = joblib.load('/content/scaler.pkl')
pencoder = joblib.load('/content/encoder.pkl')
pmodel = load_model('/content/hp_nn_model.keras')

# Function to predict
#@st.cache_resource
def predict(imodel, _iscaler, _iencoder, ifeatures):
    # encode the features
    #for ft in ifeatures:
    #  if ifeatures[ft].
    print(ifeatures, type(ifeatures))
    ind_cols =  ["bedrooms", "bathrooms", "area", "zipcode"]
    cat_col = ['zipcode']
    num_col = ['bedrooms','bathrooms','area']
    tmp_df = pd.DataFrame(ifeatures,columns=ind_cols)
    encoded_features = pd.DataFrame(_iencoder.transform(tmp_df[cat_col]))

    # Scale the features
    scaled_features = pd.DataFrame(_iscaler.transform(tmp_df[num_col]))
    print(type(scaled_features),scaled_features)
    print(type(encoded_features),encoded_features)
    tmp_fin_data = pd.concat([scaled_features,encoded_features], axis=1)
    print(tmp_fin_data)

    # Predict using the Neural Network model
    prediction = imodel.predict(tmp_fin_data)

    return prediction

#fp = '/content/drive/MyDrive/Customer Data.csv'
fp='/content/HousesInfo.txt'
cols = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(fp, sep=" ", header=None, names=cols)

# Loop through each column and display a selectbox with the minimum value as the default
selected_values = {}
for column in df.drop(columns=['price']).columns:
    # Get the minimum value of the column
    min_value = df[column].min()

    if column=='zipcode':

      # Display a selectbox for the column with the minimum value as the default
      selected_values[column] = st.selectbox(
          f'Select a value for {column}',
          df[column].unique().tolist(),
          index=df[column].tolist().index(min_value)  # Set the default to the minimum value
      )

    else:
      # Check if the column's dtype is an integer type
      if np.issubdtype(df[column].dtype, np.integer):
          step = 1  # Integer step
      else:
          step = 0.5  # Float step

      # Display a number input box for the column with the minimum value as the default
      selected_values[column] = st.number_input(
          f'Select a value for {column}',
        min_value=min_value,  # minimum allowed value
        value=min_value,      # default value set to the minimum value
        #step=1 if pd.api.types.is_integer_dtype(df[column]) else 0.01  # Step size depending on data type
        #step=1 if isinstance(df[column].dtype,'Int64') else 0.01
        step = step
      )

# Add a submit button
if st.button('Submit'):
    # Load your trained model (replace 'your_model.joblib' with your model's file path)
    #model = joblib.load('kmeans_model.pkl')
    #model = load_model('/content/hp_nn_model.keras')

    # Prepare the data for prediction (ensure the data matches the model's expected input format)
    input_data = [list(selected_values.values())]  # Convert selected values to list for prediction
    #print(input_data)
    # Use the model to predict
    #prediction = model.predict(pd.DataFrame(input_data))
    # Get the prediction
    prediction = predict(pmodel, pscaler, pencoder, input_data)

    # Display the prediction result
    st.subheader("Prediction:")
    st.write(f"The predicted house price is: {prediction}")  # Assuming the model returns a single prediction

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501